# Notebook 05 — Graph Structure Analysis (Supplementary)

**Paper:** *Graph-Structured Reinforcement Learning for Scientific Hypothesis Generation in Materials Design*

This notebook provides a **structural analysis of the directed knowledge graphs** produced by Graph-PRefLexOR models inside the `<graph_json>` reasoning phase. It validates graph well-formedness, computes structural metrics, and explores correlations between graph complexity and reasoning quality scores.

**Input data:** `data/results/graph_*b_data_eval_all.jsonl`  
**Output figures:** `figures/graph_analysis_*.png`  
**Prerequisites:** `pip install networkx scipy pandas matplotlib seaborn`

## 1. Imports and Configuration

In [ ]:
import json
import collections
import itertools
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from scipy.stats import kruskal, pearsonr

DATA_DIR    = Path("../data/results")
FIGURES_DIR = Path("../figures")
FIGURES_DIR.mkdir(exist_ok=True)

# Models to compare
MODELS = {
    "8B":   DATA_DIR / "graph_8b_data_eval_all.jsonl",
    "3B":   DATA_DIR / "graph_3b_data_eval_all.jsonl",
    "1.7B": DATA_DIR / "graph_1_7b_data_eval_all.jsonl",
}

SCORE_COLS = [
    "reasoning_quality_score",
    "intellectual_depth_score",
    "reasoning_traceability_score",
    "overall_score",
]


## 2. Load and Parse Graph JSON

The `<graph_json>` field in each model response contains a validated JSON knowledge graph. We parse it and compute structural graph metrics for each response.

In [ ]:
def parse_graph_json(val):
    """Parse teacher_graph_json (string or dict) into a Python dict."""
    if isinstance(val, str):
        try:
            return json.loads(val)
        except (json.JSONDecodeError, ValueError):
            return None
    if isinstance(val, dict):
        return val
    return None


def graph_metrics(gj: dict) -> dict:
    """Compute structural metrics for a single knowledge graph.

    Returns a dict with keys:
        n_nodes, n_edges, density, avg_degree, has_cycle, n_components.
    """
    if gj is None:
        return {k: np.nan for k in ["n_nodes","n_edges","density",
                                    "avg_degree","has_cycle","n_components"]}
    nodes = gj.get("nodes", [])
    edges = gj.get("edges", [])
    n, e  = len(nodes), len(edges)

    G = nx.DiGraph()
    G.add_nodes_from([nd.get("id","") for nd in nodes])
    G.add_edges_from([(ed.get("source",""), ed.get("target","")) for ed in edges])

    density    = e / (n * (n - 1)) if n > 1 else 0
    avg_degree = (2 * e / n) if n > 0 else 0
    has_cycle  = int(not nx.is_directed_acyclic_graph(G))
    n_comp     = nx.number_weakly_connected_components(G)

    return {"n_nodes": n, "n_edges": e, "density": density,
            "avg_degree": avg_degree, "has_cycle": has_cycle,
            "n_components": n_comp}


def load_model_data(path: Path) -> pd.DataFrame:
    """Load a model's evaluation JSONL and parse graph_json into metric columns."""
    df = pd.read_json(path, lines=True)

    # Parse graph JSON
    if "teacher_graph_json" in df.columns:
        df["graph_json_parsed"] = df["teacher_graph_json"].apply(parse_graph_json)
    elif "graph_json" in df.columns:
        df["graph_json_parsed"] = df["graph_json"].apply(parse_graph_json)
    else:
        df["graph_json_parsed"] = None

    # Expand structural metrics into columns
    metrics_df = df["graph_json_parsed"].apply(
        lambda gj: pd.Series(graph_metrics(gj))
    )
    df = pd.concat([df, metrics_df], axis=1)

    # Drop rows where the judge gave overall_score of 0 (unevaluated/invalid)
    if "overall_score" in df.columns:
        df = df[df["overall_score"] > 0].copy()

    return df


# Load all three Graph-PRefLexOR model results
raw = {name: load_model_data(path) for name, path in MODELS.items()}
for name, df in raw.items():
    print(f"{name}: {len(df)} valid responses")


## 3. Graph Validity Audit

Check how often the model produces structurally valid graphs: parseable JSON, non-empty node/edge sets, and no dangling edges (edges that reference non-existent nodes).

In [ ]:
def audit_graph(gj: dict) -> dict:
    """Return per-graph validity flags for a parsed graph JSON."""
    if gj is None:
        return dict(parse_ok=False, has_nodes=False, has_edges=False,
                    dangling_count=0, is_valid=False)

    nodes    = {nd.get("id","") for nd in gj.get("nodes", [])}
    edges    = gj.get("edges", [])
    dangling = sum(1 for e in edges
                   if e.get("source","") not in nodes or e.get("target","") not in nodes)

    return dict(
        parse_ok      = True,
        has_nodes     = len(nodes) > 0,
        has_edges     = len(edges) > 0,
        dangling_count= dangling,
        is_valid      = len(nodes) > 0 and len(edges) > 0 and dangling == 0,
    )


# Compute validity flags for each model
audit_results = {}
for name, df in raw.items():
    adf = df["graph_json_parsed"].apply(audit_graph).apply(pd.Series)
    audit_results[name] = {
        "total":     len(adf),
        "parse_ok":  adf["parse_ok"].sum(),
        "has_nodes": adf["has_nodes"].sum(),
        "has_edges": adf["has_edges"].sum(),
        "dangling":  (adf["dangling_count"] > 0).sum(),
        "fully_valid": adf["is_valid"].sum(),
    }

# Print summary table
print(f"{'Model':<8} {'Total':>6} {'Parsed':>8} {'HasNodes':>10} "
      f"{'HasEdges':>10} {'Dangling':>10} {'Valid':>8}")
for name, r in audit_results.items():
    print(f"{name:<8} {r['total']:>6} {r['parse_ok']:>8} {r['has_nodes']:>10} "
          f"{r['has_edges']:>10} {r['dangling']:>10} {r['fully_valid']:>8}")


# Bar chart: validity across models
fig, ax = plt.subplots(figsize=(10, 4), dpi=150)
x      = np.arange(len(audit_results))
width  = 0.15
cats   = ["parse_ok", "has_nodes", "has_edges", "dangling", "fully_valid"]
labels = ["Parsed", "Has Nodes", "Has Edges", "Dangling Edges", "Fully Valid"]

for i, (cat, label) in enumerate(zip(cats, labels)):
    vals = [r[cat] / r["total"] * 100 for r in audit_results.values()]
    ax.bar(x + (i - 2) * width, vals, width * 0.9, label=label)

ax.set_xticks(x)
ax.set_xticklabels(list(audit_results.keys()))
ax.set_ylabel("Percentage of Responses (%)")
ax.set_title("Graph Validity Audit Across Model Scales")
ax.legend(fontsize=8, bbox_to_anchor=(1, 1))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "graph_validity_audit.png", dpi=200, bbox_inches="tight")
plt.show()


## 4. Graph Attribute Comparison Across Scales

Violin plots compare the distribution of structural graph metrics (node count, edge count, density, average degree) across the 8B, 3B, and 1.7B models.

In [ ]:
METRIC_LABELS = {
    "n_nodes":    "Node Count",
    "n_edges":    "Edge Count",
    "density":    "Graph Density",
    "avg_degree": "Avg. Degree",
}

# Collect metric data for all models
metrics_all = {}
for name, df in raw.items():
    metrics_all[name] = df[[*METRIC_LABELS.keys()]].dropna()

# Violin grid
fig, axes = plt.subplots(2, 2, figsize=(12, 8), dpi=150)
axes = axes.flatten()

for i, (metric, label) in enumerate(METRIC_LABELS.items()):
    ax  = axes[i]
    data   = [metrics_all[n][metric].values for n in MODELS.keys()]
    labels = list(MODELS.keys())

    ax.violinplot(data, positions=range(len(labels)), showmedians=True, showextrema=False)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_title(label, fontsize=10)
    ax.set_ylabel("Value")
    ax.spines[["top", "right"]].set_visible(False)

    # Kruskal-Wallis test across models
    stat, p = kruskal(*data)
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    ax.set_xlabel(f"Kruskal-Wallis p={p:.3f} {sig}", fontsize=8)

plt.suptitle("Graph Structural Metrics: 8B vs 3B vs 1.7B", fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "graph_attrs_violin.png", dpi=200, bbox_inches="tight")
plt.show()


## 5. Correlation: Graph Attributes × Reasoning Quality

Pearson correlation between each structural graph metric and Claude-assigned scores. Tests whether richer graph structure correlates with higher reasoning quality.

In [ ]:
# Merge eval scores with structural metrics
corr_results = {}
for name, df in raw.items():
    if not all(c in df.columns for c in SCORE_COLS[:3]):
        print(f"  [skip] {name} — missing score columns")
        continue
    metric_cols = list(METRIC_LABELS.keys())
    sub = df[metric_cols + SCORE_COLS].dropna()

    # Correlation matrix: metrics × scores
    r_matrix = pd.DataFrame(index=metric_cols, columns=SCORE_COLS, dtype=float)
    p_matrix = pd.DataFrame(index=metric_cols, columns=SCORE_COLS, dtype=float)
    for metric in metric_cols:
        for score in SCORE_COLS:
            r, p = pearsonr(sub[metric], sub[score])
            r_matrix.loc[metric, score] = r
            p_matrix.loc[metric, score] = p

    corr_results[name] = (r_matrix, p_matrix)
    print(f"\n{name} — Pearson r (metrics × scores):")
    print(r_matrix.round(3).to_string())


# Heatmap comparison across scales
fig, axes = plt.subplots(1, 3, figsize=(15, 4), dpi=150)
for ax, (name, (r_mat, _)) in zip(axes, corr_results.items()):
    sns.heatmap(r_mat.astype(float), annot=True, fmt=".2f", center=0,
                cmap="coolwarm", vmin=-0.5, vmax=0.5, ax=ax,
                cbar=(ax is axes[-1]))
    ax.set_title(f"{name} Model", fontsize=10)
    ax.set_yticklabels([METRIC_LABELS.get(m, m) for m in r_mat.index], rotation=0, fontsize=8)
    ax.set_xticklabels(["RQ", "ID", "RT", "Overall"], rotation=30, fontsize=8)

plt.suptitle("Pearson r: Graph Attributes × Reasoning Quality Scores", fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "graph_attrs_corr_heatmaps.png", dpi=200, bbox_inches="tight")
plt.show()


## 6. Phase Length and Allocation

Analyse how word-count is distributed across the four reasoning phases (`<brainstorm>`, `<graph>`, `<patterns>`, `<synthesis>`) for each model scale.

In [ ]:
def extract_phase_word_counts(raw_text: str) -> dict:
    """Count words in each reasoning phase of a Graph-PRefLexOR response."""
    def word_count(text):
        return len(text.split())

    import re
    result = {}
    for phase in ["brainstorm", "graph", "patterns", "synthesis"]:
        pattern = re.compile(f"<{phase}>(.*?)</{phase}>", re.DOTALL)
        m = pattern.search(raw_text)
        result[phase] = word_count(m.group(1)) if m else 0
    return result


# Compute phase word counts for all models
phase_data = {}
for name, df in raw.items():
    col = "raw_output" if "raw_output" in df.columns else "chosen"
    if col not in df.columns:
        continue
    wc = df[col].apply(extract_phase_word_counts).apply(pd.Series)
    phase_data[name] = wc

# Stacked fraction bar — how is the thinking budget allocated per phase?
fig, ax = plt.subplots(figsize=(8, 5), dpi=150)
palette = sns.color_palette("Set2", 4)
model_names = list(phase_data.keys())
x = np.arange(len(model_names))
bottom = np.zeros(len(model_names))

for i, phase in enumerate(["brainstorm", "graph", "patterns", "synthesis"]):
    fracs = []
    for name in model_names:
        wc = phase_data[name]
        total = wc.sum(axis=1).replace(0, np.nan)
        fracs.append((wc[phase] / total).mean())
    ax.bar(x, fracs, bottom=bottom, label=f"<{phase}>", color=palette[i])
    bottom += np.array(fracs)

ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.set_ylabel("Mean Phase Fraction")
ax.set_title("Phase Allocation of Reasoning Budget Across Scales")
ax.legend(fontsize=8, bbox_to_anchor=(1, 1))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "phase_allocation_bar.png", dpi=200, bbox_inches="tight")
plt.show()
